In [ ]:
import pandas as pd
import numpy as np
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
try:
    from xgboost import XGBRegressor  # type: ignore
except ModuleNotFoundError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "xgboost"])
    from xgboost import XGBRegressor  # type: ignore
import warnings
import sys
import subprocess
warnings.filterwarnings("ignore")

# 1. 加载数据（只需修改这里↓↓↓）
data_path = "E:\input_results\训练集_原始数据.xlsx"  
data = pd.read_excel(data_path)

# 确保数据中包含Infection列
if 'Infection' not in data.columns:
    raise ValueError("数据中必须包含'Infection'列作为标签")

# 2. 找出所有完整列（无缺失值的列），不包括Infection列
complete_cols = [col for col in data.columns if (data[col].isnull().sum() == 0) and (col != 'Infection')]
print("完整列有:", complete_cols)

if not complete_cols:
    raise ValueError("没有找到完整列！")

# 3. 对每个完整列随机删除10%数据（制造缺失）
np.random.seed(42)  # 固定随机种子
data_with_missing = data.copy()
missing_records = pd.DataFrame()  # 记录被删除的真实值

for col in complete_cols:
    missing_idx = np.random.choice(
        data.index,
        size=int(len(data)*0.1),
        replace=False
    )
    # 记录被删除的值
    missing_records = pd.concat([
        missing_records,
        pd.DataFrame({
            '列名': col,
            '行索引': missing_idx,
            '真实值': data.loc[missing_idx, col]
        })
    ])
    # 设置缺失值
    data_with_missing.loc[missing_idx, col] = np.nan

ModuleNotFoundError: No module named 'xgboost'

In [ ]:
# 4. XGBoost插值（包含Infection作为预测因子）
# 4. XGBoost插值（10次随机种子）

# Phase 1

n_estimators_list = [
    50,
    100,
    200
]

learning_rates = [
    0.01,
    0.04,
    0.07,
    0.10,
    0.13,
    0.16,
    0.19,
    0.22,
    0.25,
    0.28
]

phase1_results = []

for n_est in n_estimators_list:

    for lr in learning_rates:

        print(
            f"\nTesting "
            f"n_estimators={n_est}, "
            f"learning_rate={lr}"
        )

        imputer = IterativeImputer(

            estimator=XGBRegressor(
                n_estimators=n_est,
                learning_rate=lr,
                random_state=42,
                n_jobs=-1
            ),

            max_iter=12,
            random_state=42
        )

        imputed_data = imputer.fit_transform(
            data_with_missing
        )

        imputed_df = pd.DataFrame(
            imputed_data,
            columns=data.columns
        )

        nrmse_list = []

        for col in complete_cols:

            col_records = missing_records[
                missing_records["列名"] == col
            ]

            y_true = col_records["真实值"]

            y_pred = imputed_df.loc[
                col_records["行索引"],
                col
            ]

            rmse = np.sqrt(
                np.mean(
                    (y_true - y_pred) ** 2
                )
            )

            nrmse = rmse / (
                y_true.max() - y_true.min()
            )

            nrmse_list.append(nrmse)

        mean_nrmse = np.mean(
            nrmse_list
        )

        phase1_results.append({
            "n_estimators": n_est,
            "learning_rate": lr,
            "Mean_NRMSE": mean_nrmse
        })

phase1_df = pd.DataFrame(
    phase1_results
)

phase1_df = phase1_df.sort_values(
    "Mean_NRMSE"
)

phase1_df.to_excel(
    "Phase1_All_Results.xlsx",
    index=False
)

print("\nTOP 3")

print(
    phase1_df.head(3)
)


In [ ]:
# Phase 2

top3_params = [

    (100, 0.16),

    (200, 0.13),

    (100, 0.19)

]

phase2_results = []

for n_est, lr in top3_params:

    print("\n===================")
    print(
        f"Testing "
        f"n_estimators={n_est}, "
        f"lr={lr}"
    )
    print("===================")

    run_nrmse = []

    for seed in range(1,11):

        print(f"Seed={seed}")

        imputer = IterativeImputer(

            estimator=XGBRegressor(

                n_estimators=n_est,

                learning_rate=lr,

                random_state=seed,

                n_jobs=-1
            ),

            max_iter=12,

            random_state=seed
        )

        imputed_data = imputer.fit_transform(
            data_with_missing
        )

        imputed_df = pd.DataFrame(
            imputed_data,
            columns=data.columns
        )

        nrmse_list = []

        for col in complete_cols:

            col_records = missing_records[
                missing_records["列名"] == col
            ]

            y_true = col_records["真实值"]

            y_pred = imputed_df.loc[
                col_records["行索引"],
                col
            ]

            rmse = np.sqrt(
                np.mean(
                    (y_true - y_pred) ** 2
                )
            )

            nrmse = rmse / (
                y_true.max() - y_true.min()
            )

            nrmse_list.append(nrmse)

        mean_nrmse = np.mean(
            nrmse_list
        )

        run_nrmse.append(
            mean_nrmse
        )

    overall_mean = np.mean(
        run_nrmse
    )

    overall_sd = np.std(
        run_nrmse,
        ddof=1
    )

    ci_low = np.percentile(
        run_nrmse,
        2.5
    )

    ci_high = np.percentile(
        run_nrmse,
        97.5
    )

    phase2_results.append({

        "n_estimators": n_est,

        "learning_rate": lr,

        "Mean_NRMSE": overall_mean,

        "SD_NRMSE": overall_sd,

        "CI_Low": ci_low,

        "CI_High": ci_high

    })

phase2_df = pd.DataFrame(
    phase2_results
)

phase2_df = phase2_df.sort_values(
    "Mean_NRMSE"
)

phase2_df.to_excel(
    "Phase2_Final_Ranking.xlsx",
    index=False
)

print("\n===================")
print("FINAL RESULT")
print("===================")

print(
    phase2_df
)